# Schedule Retry Contract v1

本实验验证 `src/schedule` 的统一 retry 契约：数据更新、模型刷新、production health 和 paper execution 不再各自复制 retry loop，而是复用 `run_task_with_retries`。

In [1]:
from pathlib import Path
import json
import sys

current = Path.cwd().resolve()
repo_root = next(candidate for candidate in (current, *current.parents) if (candidate / "src").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.schedule import run_task_with_retries

OUTPUT_DIR = repo_root / "research" / "schedule_retry_contract_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 成功样本

先模拟一次瞬时失败，再成功，验证 payload 中保留 attempts 和 retry error history。

In [2]:
calls = {"count": 0}

def flaky_task():
    calls["count"] += 1
    if calls["count"] == 1:
        raise RuntimeError("temporary provider failure")
    return {"schema_version": "unit.retry_payload.v1", "status": "ok"}

success_payload = run_task_with_retries(flaky_task, retries=2, failure_message="unit retry failed")
success_payload

{'schema_version': 'unit.retry_payload.v1',
 'status': 'ok',
 'attempts': 2,
 'retry_schema_version': 'schedule.retry_attempts.v1',
 'retry_error_count': 1,
 'retry_errors': [{'attempt': 1,
   'error_type': 'RuntimeError',
   'message': 'temporary provider failure'}]}

## 失败样本

超过 retry budget 后抛出最后一次错误；上层 `run_job_once` 会负责转成 failed JobResult 和通知。

In [3]:
failure_calls = {"count": 0}

def failing_task():
    failure_calls["count"] += 1
    raise RuntimeError(f"permanent failure {failure_calls['count']}")

failure_message = ""
try:
    run_task_with_retries(failing_task, retries=1, failure_message="unit retry failed")
except RuntimeError as exc:
    failure_message = str(exc)

failure_message

'permanent failure 2'

## 断言

统一契约要求成功 payload 可解释，失败路径保留最后错误。

In [4]:
assert success_payload["status"] == "ok"
assert success_payload["attempts"] == 2
assert success_payload["retry_schema_version"] == "schedule.retry_attempts.v1"
assert success_payload["retry_error_count"] == 1
assert success_payload["retry_errors"] == [
    {"attempt": 1, "error_type": "RuntimeError", "message": "temporary provider failure"}
]
assert failure_calls["count"] == 2
assert failure_message == "permanent failure 2"

In [5]:
analysis_summary = {
    "schema_version": "research.schedule_retry_contract_v1.summary.v1",
    "experiment": "schedule_retry_contract_v1",
    "core_contract": "schedule.retry_attempts.v1",
    "success_attempts": success_payload["attempts"],
    "success_retry_error_count": success_payload["retry_error_count"],
    "failure_attempts": failure_calls["count"],
    "failure_message": failure_message,
    "promotion_decision": "promote_to_src_schedule_jobs",
    "risk_note": "该契约统一 retry 元数据，不替代具体 provider/API 的错误分类。",
}

summary_path = OUTPUT_DIR / "analysis_summary.json"
summary_path.write_text(json.dumps(analysis_summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
analysis_summary

{'schema_version': 'research.schedule_retry_contract_v1.summary.v1',
 'experiment': 'schedule_retry_contract_v1',
 'core_contract': 'schedule.retry_attempts.v1',
 'success_attempts': 2,
 'success_retry_error_count': 1,
 'failure_attempts': 2,
 'failure_message': 'permanent failure 2',
 'promotion_decision': 'promote_to_src_schedule_jobs',
 'risk_note': '该契约统一 retry 元数据，不替代具体 provider/API 的错误分类。'}

## 中文结论

- 做了什么：新增并验证 `run_task_with_retries`，把 schedule pipeline 中重复的 retry loop 收敛到 `src/schedule/jobs.py`，成功 payload 统一带 `attempts`、`retry_schema_version`、`retry_error_count` 和 `retry_errors`。
- 核心指标：瞬时失败样本第二次成功，`attempts=2`、`retry_error_count=1`；永久失败样本在 retry budget 后执行 2 次，并抛出最后错误 `permanent failure 2`。
- regime 差异：本实验不是市场 regime 回测，而是运行状态 regime；transient failure 进入 retry 后成功，permanent failure 超过预算后交给 `run_job_once` 转为 failed job。
- 是否晋升：晋升到 `src/schedule/jobs.py`，因为数据更新、模型刷新、production health 和 paper execution 都需要同一套 retry 元数据契约。
- 风险：该契约统一 retry 元数据，不负责判断 provider/API 错误是否可重试；错误分类仍应由具体 adapter 或 pipeline 决定。
- 下一步：把 retry error history 展示到详细通知或 dashboard schedule health 面板，便于区分 transient 和 permanent 失败。